# REPSOL Model Training (70/15/15)

**EfficientNet-02**

This notebook runs the full pipeline: 
1. Configure training hyperparameters.
2. Verify spectrogram tensor files in train/val/test.
3. Train EfficientNet with live progress bars.
4. Evaluate on validation and test sets.
5. Print detailed classification report and confusion matrix.

In [1]:
from pathlib import Path
import torch

# ===== Hyperparameters (edit these) =====
BATCH_SIZE = 8  # Reduced from 16 to 8 for CPU memory stability
EPOCHS = 15
LEARNING_RATE = 1e-3
PATIENCE = 4
MODEL_NAME = "efficientnet"

# ===== Paths =====
PROJECT_ROOT = Path(r"D:\Work\Internships\INMAR\REPSOL")
SPECTROGRAM_DIR = PROJECT_ROOT / "Data" / "Spectrograms"
OUTPUT_DIR = PROJECT_ROOT / "Models_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def next_run_path(model_name: str, suffix: str, ext: str, output_dir: Path) -> Path:
    prefix = f"{model_name}{suffix}"
    existing_numbers = [0]
    for path in output_dir.iterdir():
        if not path.is_file() or path.suffix != ext:
            continue
        stem = path.stem
        if stem == prefix:
            existing_numbers.append(0)
            continue
        if stem.startswith(prefix + "_"):
            suffix_text = stem[len(prefix) + 1 :]
            if suffix_text.isdigit():
                existing_numbers.append(int(suffix_text))
    next_num = max(existing_numbers) + 1
    return output_dir / f"{prefix}_{next_num:02d}{ext}"

CHECKPOINT_PATH = next_run_path(MODEL_NAME, "_best", ".pth", OUTPUT_DIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPECTROGRAM_DIR:", SPECTROGRAM_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("DEVICE:", DEVICE)
print("BATCH_SIZE:", BATCH_SIZE, "| EPOCHS:", EPOCHS, "| LR:", LEARNING_RATE)

PROJECT_ROOT: D:\Work\Internships\INMAR\REPSOL
SPECTROGRAM_DIR: D:\Work\Internships\INMAR\REPSOL\Data\Spectrograms
OUTPUT_DIR: D:\Work\Internships\INMAR\REPSOL\Models_output
CHECKPOINT_PATH: D:\Work\Internships\INMAR\REPSOL\Models_output\efficientnet_best_03.pth
DEVICE: cpu
BATCH_SIZE: 8 | EPOCHS: 15 | LR: 0.001


In [2]:
def count_pt_files(root):
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.pt")) if split_dir.exists() else 0
    return counts

counts = count_pt_files(SPECTROGRAM_DIR)
print("PT files by split:", counts)
print("Total:", sum(counts.values()))

assert counts["train"] > 0, "No train .pt files found."
assert counts["val"] > 0, "No val .pt files found."
assert counts["test"] > 0, "No test .pt files found."

PT files by split: {'train': 1382, 'val': 296, 'test': 297}
Total: 1975


In [3]:
# Install/verify training dependencies in the active notebook kernel
import importlib
import subprocess
import sys

required = ["torch", "torchvision", "torchaudio", "scikit-learn", "pandas", "tqdm", "numpy"]
name_map = {
    "scikit-learn": "sklearn",
}

for pkg in required:
    import_name = name_map.get(pkg, pkg.replace("-", "_"))
    try:
        importlib.import_module(import_name)
        print(f"OK: {pkg}")
    except Exception:
        print(f"Installing: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed: {pkg}")

OK: torch
OK: torchvision
OK: torchaudio
OK: scikit-learn
OK: pandas
OK: tqdm
OK: numpy


---

Training with base EfficientNet

In [4]:
import importlib
import torch

try:
    import torchvision  # noqa: F401
except Exception as e:
    raise RuntimeError(
        "torchvision is not available in this notebook kernel. "
        "Run the dependency cell right above this one, then retry."
    ) from e

import src.EfficientNet.model as model_module
import src.EfficientNet.train as train_module

model_module = importlib.reload(model_module)
train_module = importlib.reload(train_module)
Trainer = train_module.Trainer

trainer = Trainer(
    spectrogram_dir=SPECTROGRAM_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    max_epochs=EPOCHS,
    patience=PATIENCE,
    lr=LEARNING_RATE,
    device=DEVICE,
)

trainer.fit()

Train samples: 1382
Train batches: 173


Epoch 1/15 Validation: 100%|████████████████████████| 37/37 [05:25<00:00,  8.80s/batch, loss=3.1646]

Epoch 1/15 | Train Loss: 1.7626 | Val Loss: 1.5065 | Train Acc: 31.98 | Val Acc: 47.30


Saved improved checkpoint to: D:\Work\Internships\INMAR\REPSOL\Models_output\efficientnet_best_03.pth


Epoch 2/15 Validation: 100%|████████████████████████| 37/37 [05:27<00:00,  8.84s/batch, loss=3.0558]

...

Epoch 10/15 | Train Loss: 0.6203 | Val Loss: 1.2371 | Train Acc: 73.08 | Val Acc: 54.73
No improvement for 2/4 epochs



Epoch 11/15 Validation: 100%|███████████████████████| 37/37 [05:34<00:00,  9.05s/batch, loss=2.5425]

Epoch 11/15 | Train Loss: 0.5287 | Val Loss: 1.4273 | Train Acc: 76.19 | Val Acc: 52.36
No improvement for 3/4 epochs



Epoch 12/15 Validation: 100%|███████████████████████| 37/37 [05:32<00:00,  8.99s/batch, loss=2.4910]

Epoch 12/15 | Train Loss: 0.3558 | Val Loss: 1.0565 | Train Acc: 81.26 | Val Acc: 65.88
No improvement for 4/4 epochs
Early stopping: no improvement for 4 epochs.
Training finished.


In [ ]:
import importlib
import torch
import src.evaluate as eval_module

eval_module = importlib.reload(eval_module)
evaluate_model = eval_module.evaluate_model

state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
trainer.model.load_state_dict(state)

val_metrics = evaluate_model(trainer.model, trainer.val_loader, DEVICE)
test_metrics = evaluate_model(trainer.model, trainer.test_loader, DEVICE)

print("Validation Metrics")
print({
    "accuracy": round(val_metrics["accuracy"], 4),
    "precision": round(val_metrics["precision"], 4),
    "recall": round(val_metrics["recall"], 4),
    "f1": round(val_metrics["f1"], 4),
})

print("\nTest Metrics")
print({
    "accuracy": round(test_metrics["accuracy"], 4),
    "precision": round(test_metrics["precision"], 4),
    "recall": round(test_metrics["recall"], 4),
    "f1": round(test_metrics["f1"], 4),
})

In [ ]:
print("Test Classification Report:\n")
print(test_metrics["report"])

print("Test Confusion Matrix:")
print(test_metrics["confusion_matrix"])

Test Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.73      0.81        33
           1       0.43      0.50      0.46         6
           2       0.47      0.80      0.59        20
           3       0.71      0.63      0.67       106
           4       0.60      0.15      0.24        39
           5       1.00      1.00      1.00        11
           6       0.66      0.93      0.77        76
           7       0.43      0.50      0.46         6

    accuracy                           0.68       297
   macro avg       0.65      0.66      0.63       297
weighted avg       0.69      0.68      0.65       297

Test Confusion Matrix:
[[24  0  0  0  2  0  7  0]
 [ 0  3  0  1  0  0  2  0]
 [ 0  0 16  4  0  0  0  0]
 [ 2  2  7 67  2  0 23  3]
 [ 0  0 11 18  6  0  4  0]
 [ 0  0  0  0  0 11  0  0]
 [ 0  1  0  3  0  0 71  1]
 [ 0  1  0  1  0  0  1  3]]
